In [ ]:
makeData = False

In [ ]:
# ==============================================================================
# Part 0: Introduction
# ==============================================================================
# I am Siddant Bapna
# ------------------------------------------------------------------------------

print("Hello, I am sb")

In [ ]:
# ==============================================================================
# Part 1: Installation
# ==============================================================================
# This cell installs the required libs.
# ------------------------------------------------------------------------------

# !pip install -q synapseclient "monai[nibabel, tqdm]" joblib scikit-image

import subprocess
import sys
import pkg_resources

def install_if_not_installed(package):
    try:
        # Check if package is installed
        pkg_resources.get_distribution(package)
        print(f"{package} is already installed.")
    except pkg_resources.DistributionNotFound:
        print(f"{package} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# List of required packages
packages = [
    "synapseclient",
    "monai[nibabel,tqdm]",
    "joblib",
    "scikit-image"
]

for package in packages:
    install_if_not_installed(package)

In [ ]:
# ==============================================================================
# Part 2: UserSecretsClient
# ==============================================================================
# Import Kaggle UserSecretsClient for secure token access ---
# ------------------------------------------------------------------------------

if makeData : 
    print("No")

import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    stk = user_secrets.get_secret("SYNAPSE_AUTH_TOKEN")
    print("Synapse token loaded successfully from Kaggle Secrets.")
except ImportError:
    stk = os.environ.get("SYNAPSE_AUTH_TOKEN", None)
    if stk:
        print("Synapse token loaded successfully from environment variable.")
    else:
        # Fallback for local execution if secrets/env var not set.
        # Replace "PASTE_YOUR_TOKEN_HERE" with your actual token for local tests.
        stk = "PASTE_YOUR_TOKEN_HERE"
        if stk == "PASTE_YOUR_TOKEN_HERE":
            raise RuntimeError("Synapse token not found. Please set it in Kaggle Secrets (key: SYNAPSE_AUTH_TOKEN) or define the 'stk' variable manually.")
        else:
            print("Synapse token loaded from manual variable definition.")

In [ ]:
# ==============================================================================
# Part 2: Data Downloading & Unzipping
# ==============================================================================
# This section handles authentication and download of the dataset from Synapse.
# ------------------------------------------------------------------------------
import synapseclient
import zipfile
import shutil

if makeData : 
    print("No")

# --- Synapse Login ---
syn = synapseclient.Synapse()
try:
    syn.login(authToken=stk, silent=True)
    print("Synapse login successful.")
except Exception as e:
    print(f"Synapse login failed. Please ensure your authToken is correct. Error: {e}")
    raise

# --- File & Directory Setup ---
idc = ["syn51514132"]
destination_dir = '/kaggle/working/BRATS/train'
os.makedirs(destination_dir, exist_ok=True)

# --- Helper Functions for Download & Unzip ---
def unzip_data(zip_path, extract_to):
    print(f"Checking for unzipped data at {extract_to}...")
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 50:
        print("Data appears to be already unzipped. Skipping.")
        return
    print(f"Unzipping {zip_path} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Unzipping complete.")
    
    # Handle nested folders
    content_list = os.listdir(extract_to)
    if len(content_list) == 1 and os.path.isdir(os.path.join(extract_to, content_list[0])):
        nested_folder = os.path.join(extract_to, content_list[0])
        print(f"Moving contents from nested folder '{nested_folder}'...")
        for item in os.listdir(nested_folder):
            shutil.move(os.path.join(nested_folder, item), extract_to)
        os.rmdir(nested_folder)

# --- Download & Unzip Execution ---
unzipped_path = os.path.join('/kaggle/working/BRATS/train')
if os.path.exists(unzipped_path) and len(os.listdir(unzipped_path)) > 50:
    print("Dataset already downloaded and unzipped. Skipping download.")
else:
    print(f"--- Starting Download: {idc[0]} ---")
    dat = syn.get(entity=idc[0])
    unzip_data(dat.path, destination_dir)
    os.remove(dat.path)
    print(f"Deleted zip file: {dat.path}")
print("--- Data Preparation Finished ---")



In [ ]:
# ==============================================================================
# Part 3: Preprocessing (NIfTI to NPZ)
# ==============================================================================
# This optimized pipeline converts raw .nii.gz files into processed .npz arrays.
# ------------------------------------------------------------------------------

if makeData : 
    print("No")

import numpy as np
from tqdm.notebook import tqdm
import concurrent.futures
from functools import partial
import torch

from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, CropForegroundd,
    Resized, ScaleIntensityRanged, ConvertToMultiChannelBasedOnBratsClassesd, EnsureTyped
)

# --- Configuration ---
BASE_DIR = '/kaggle/working/BRATS'
UNZIPPED_DIR = os.path.join(BASE_DIR, 'train')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed')

TARGET_VOXEL_SPACING = (1.0, 1.0, 1.0)
OUTPUT_SHAPE = (128, 128, 128)
MODALITY_KEYS = ['t1c', 't1n', 't2f', 't2w']
ALL_KEYS = MODALITY_KEYS + ['seg']

os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Preprocessing Functions ---
def find_patient_files(data_dir):
    patient_files = []
    for patient_id in sorted(os.listdir(data_dir)):
        patient_folder = os.path.join(data_dir, patient_id)
        if os.path.isdir(patient_folder):
            files = {
                "t1c": os.path.join(patient_folder, f"{patient_id}-t1c.nii.gz"),
                "t1n": os.path.join(patient_folder, f"{patient_id}-t1n.nii.gz"),
                "t2f": os.path.join(patient_folder, f"{patient_id}-t2f.nii.gz"),
                "t2w": os.path.join(patient_folder, f"{patient_id}-t2w.nii.gz"),
                "seg": os.path.join(patient_folder, f"{patient_id}-seg.nii.gz"),
                "id": patient_id
            }
            if all(os.path.exists(f) for k, f in files.items() if k != "id"):
                patient_files.append(files)
    return patient_files


monai_preprocess_pipeline = Compose([
    LoadImaged(keys=ALL_KEYS, image_only=True, ensure_channel_first=True),
    ConvertToMultiChannelBasedOnBratsClassesd(keys='seg'),
    Spacingd(keys=ALL_KEYS, pixdim=TARGET_VOXEL_SPACING, mode=["bilinear"] * 4 + ["nearest"]),
    ScaleIntensityRanged(keys=MODALITY_KEYS, a_min=0.0, a_max=1400.0, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=ALL_KEYS, source_key='t1c', margin=10),
    Resized(keys=ALL_KEYS, spatial_size=OUTPUT_SHAPE, mode=["area"] * 4 + ["nearest"]),
    EnsureTyped(keys=ALL_KEYS, dtype=torch.float16) # Use float16 to save disk space
])

def preprocess_and_save(patient_data, output_dir):
    try:
        processed_data = monai_preprocess_pipeline(patient_data)
        final_image = torch.cat([processed_data[key] for key in MODALITY_KEYS], dim=0)
        final_mask = processed_data['seg']
        output_filepath = os.path.join(output_dir, f"{patient_data['id']}.npz")
        np.savez_compressed(output_filepath, image=final_image.numpy(), mask=final_mask.numpy().astype(np.uint8))
    except Exception as e:
        return f"Failed {patient_data['id']}: {e}"
    return None

# --- Main Preprocessing Execution ---
patient_list = find_patient_files(UNZIPPED_DIR)
if not os.listdir(PROCESSED_DIR):
    print(f"Found {len(patient_list)} patients to preprocess.")
    process_func = partial(preprocess_and_save, output_dir=PROCESSED_DIR)
    with concurrent.futures.ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
        results = list(tqdm(executor.map(process_func, patient_list), total=len(patient_list), desc="Preprocessing"))
    print("\nPreprocessing complete!")
else:
    print("Processed data already exists. Skipping preprocessing.")

In [ ]:
# ==============================================================================
# Part 4: Data Integrity Check & K-Fold Split
# ==============================================================================

from sklearn.model_selection import KFold

if makeData : 
    print("No")

VALID_FILES = sorted([os.path.join(PROCESSED_DIR, f) for f in os.listdir(PROCESSED_DIR) if f.endswith('.npz') and os.path.getsize(os.path.join(PROCESSED_DIR, f)) > 0])
print(f"\nFound {len(VALID_FILES)} valid patient files.")

N_SPLITS = 5
FOLD_TO_RUN = 0 # Currently at zero/no fold can take (0-4)
RANDOM_STATE = 42

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
train_indices, val_indices = list(kf.split(VALID_FILES))[FOLD_TO_RUN]
train_files = [VALID_FILES[i] for i in train_indices]
val_files = [VALID_FILES[i] for i in val_indices]

print(f"\n--- FOLD {FOLD_TO_RUN}/{N_SPLITS-1} ---")
print(f"Training set size: {len(train_files)} | Validation set size: {len(val_files)}")



In [ ]:
# ==============================================================================
# Part 5: Dataset and DataLoaders
# ==============================================================================

from torch.utils.data import Dataset, DataLoader
from monai.transforms import Compose, RandFlipd, RandRotate90d, Rand3DElasticd, RandScaleIntensityd

class BraTSDataset(Dataset):
    def __init__(self, file_paths, augment=False):
        self.file_paths = file_paths
        self.augment = augment
        if self.augment:
            self.transform = Compose([
                RandFlipd(keys=["image", "mask"], prob=0.5, spatial_axis=0),
                RandFlipd(keys=["image", "mask"], prob=0.5, spatial_axis=1),
                RandFlipd(keys=["image", "mask"], prob=0.5, spatial_axis=2),
                RandRotate90d(keys=["image", "mask"], prob=0.5, max_k=3),
                RandScaleIntensityd(keys="image", factors=0.1, prob=0.5),
                Rand3DElasticd(keys=["image", "mask"], sigma_range=(3, 5), magnitude_range=(10, 30),
                               prob=0.2, mode=('bilinear', 'nearest'), padding_mode='zeros')
            ])

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        with np.load(self.file_paths[idx]) as data:
            image = torch.from_numpy(data['image'].astype(np.float32))
            mask = torch.from_numpy(data['mask'].astype(np.float32))
        if self.augment:
            data_dict = self.transform({"image": image, "mask": mask})
            return data_dict["image"], data_dict["mask"]
        return image, mask

train_dataset = BraTSDataset(file_paths=train_files, augment=True)
val_dataset = BraTSDataset(file_paths=val_files, augment=False)

BATCH_SIZE = 2
NUM_WORKERS = 2 # Set to 0 for debugging if you get DataLoader errors

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0))


In [ ]:
# ==============================================================================
# Part 6: Model, Loss, Optimizer, and Scheduler
# ==============================================================================
# Switched to AttentionUnet with safer channel sizes for Kaggle Compute
# Model's forward pass does not have a final sigmoid activation ---
# ------------------------------------------------------------------------------

import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from monai.networks.nets import AttentionUnet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- MODIFIED: Reduced initial channels to prevent OOM errors. Increase if memory permits. ---
model = AttentionUnet(
    spatial_dims=3, in_channels=4, out_channels=3,
    channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
).to(device)

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        logits, targets = logits.view(-1), targets.view(-1)
        intersection = (logits * targets).sum()
        sum_of_sets = logits.sum() + targets.sum()
        dice_coeff = (2. * intersection + self.smooth) / (sum_of_sets + self.smooth)
        return 1 - dice_coeff


class DiceBCELoss(nn.Module):
    def __init__(self, weight_dice=0.5, weight_bce=0.5):
        super().__init__()
        self.dice_loss = DiceLoss()
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.weight_dice = weight_dice
        self.weight_bce = weight_bce
    def forward(self, logits, targets):
        dice = self.dice_loss(torch.sigmoid(logits), targets)
        bce = self.bce_loss(logits, targets)
        return self.weight_dice * dice + self.weight_bce * bce

LEARNING_RATE = 1e-4
NUM_EPOCHS = 20
PATIENCE = 7    # For early stopping

criterion = DiceBCELoss().to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
poly_lambda = lambda epoch: (1 - epoch / NUM_EPOCHS) ** 0.9
scheduler = LambdaLR(optimizer, lr_lambda=poly_lambda)

# AMP GradScaler for mixed-precision training
scaler = torch.cuda.amp.GradScaler()


In [ ]:
# ==============================================================================
# Part 7: Training & Validation Loop with AMP and Early Stopping
# ==============================================================================
# Integrated AMP, scheduler, early stopping, and detailed metrics
# Checkpointing now saves a full dictionary for resuming
# ------------------------------------------------------------------------------

import time

BEST_MODEL_PATH = f'/kaggle/working/best_model_fold_{FOLD_TO_RUN}.pth'

def dice_score_per_class(preds, targets, smooth=1e-6):
    preds = torch.sigmoid(preds) > 0.5
    dice_scores = []
    for i in range(preds.shape[1]): # Iterate over channels (classes)
        pred_flat = preds[:, i, ...].contiguous().view(-1)
        target_flat = targets[:, i, ...].contiguous().view(-1)
        intersection = (pred_flat * target_flat).sum()
        union = pred_flat.sum() + target_flat.sum()
        dice = (2. * intersection + smooth) / (union + smooth)
        dice_scores.append(dice)
    return torch.stack(dice_scores)

def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(loader, desc="Training", leave=False)
    for images, masks in progress_bar:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    return running_loss / len(loader)

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_dice_scores = []
    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Validating", leave=False):
            images, masks = images.to(device), masks.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
            dice = dice_score_per_class(outputs.cpu(), masks.cpu())
            all_dice_scores.append(dice)
            running_loss += loss.item()
    
    avg_dice = torch.stack(all_dice_scores).mean(0)
    return running_loss / len(loader), avg_dice

# --- Main Training Loop ---
best_val_dice = -1.0
no_improve_epochs = 0
start_time = time.time()

print(f"\nStarting Training for Fold {FOLD_TO_RUN}...")
for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_dice_per_class = validate_one_epoch(model, val_loader, criterion, device)
    scheduler.step()
    
    avg_val_dice = val_dice_per_class.mean().item()
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Dice (Avg): {avg_val_dice:.4f} [WT: {val_dice_per_class[0]:.4f}, TC: {val_dice_per_class[1]:.4f}, ET: {val_dice_per_class[2]:.4f}]")
    
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        no_improve_epochs = 0
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'best_val_dice': best_val_dice
        }, BEST_MODEL_PATH)
        print(f"✅ New best model saved (Dice: {best_val_dice:.4f})")
    else:
        no_improve_epochs += 1
    
    if no_improve_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1} (no improvement in {PATIENCE} epochs).")
        break

print(f"\n✅ Training Finished in {(time.time() - start_time) / 60:.2f} minutes.")




In [ ]:
BEST_MODEL_PATH = "/kaggle/input/sb23-2/best_model_fold_0.pth"

In [ ]:
# ==============================================================================
# Part 8: Post-Processing & Visualization
# ==============================================================================

import matplotlib.pyplot as plt
from skimage.measure import label

def remove_small_lesions(pred_mask_np, min_size_map):
    processed_mask = np.zeros_like(pred_mask_np)
    for c in range(pred_mask_np.shape[0]):
        channel_mask = pred_mask_np[c]
        if np.sum(channel_mask) > 0:
            labeled_mask = label(channel_mask)
            min_size = min_size_map.get(c, 50)
            for region_label in range(1, np.max(labeled_mask) + 1):
                if np.sum(labeled_mask == region_label) > min_size:
                    processed_mask[c][labeled_mask == region_label] = 1
    return processed_mask

model.load_state_dict(torch.load(BEST_MODEL_PATH)['model_state_dict'])
model.eval()

def visualize_prediction(dataset_idx):
    image, true_mask = val_dataset[dataset_idx]
    input_tensor = image.unsqueeze(0).to(device)
    
    with torch.no_grad(), torch.cuda.amp.autocast():
        prediction_logits = model(input_tensor)

    pred_mask_raw = (torch.sigmoid(prediction_logits).cpu().squeeze(0) > 0.5).numpy()
    
    min_lesion_sizes = {0: 100, 1: 75, 2: 50} # WT, TC, ET - tune these on your val set
    pred_mask_postprocessed = remove_small_lesions(pred_mask_raw, min_lesion_sizes)

    # --- CORRECTED: Slicing the last dimension (Depth) for an axial view ---
    slice_idx = image.shape[-1] // 2
    
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    fig.suptitle(f'Sample {dataset_idx} - Central Axial Slice ({slice_idx})', fontsize=16)
    
    titles = ["Whole Tumor (WT)", "Tumor Core (TC)", "Enhancing Tumor (ET)"]
    for i, title in enumerate(titles):
        axes[i, 0].imshow(image[0, :, :, slice_idx], cmap='gray'); axes[i, 0].set_title(f'Input T1c\n({title})')
        axes[i, 1].imshow(true_mask[i, :, :, slice_idx], cmap='viridis'); axes[i, 1].set_title(f'Ground Truth\n({title})')
        axes[i, 2].imshow(pred_mask_postprocessed[i, :, :, slice_idx], cmap='viridis'); axes[i, 2].set_title(f'Prediction\n({title})')
        for ax in axes[i]: ax.axis('off')
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

print("\n--- Visualizing a few validation samples with post-processing ---")
visualize_prediction(5)
visualize_prediction(15)
visualize_prediction(25)

In [ ]:
# ==============================================================================
# Part 1: Setup for Inference
# ==============================================================================
# This cell prepares the environment for running inference on your trained model.
# It defines the model architecture, helper functions, and loads the saved weights.
# ------------------------------------------------------------------------------
import os
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from skimage import measure
from tqdm import tqdm

# Ensure necessary libraries are available
!pip install -q "monai[nibabel, tqdm]" scikit-image plotly

from monai.networks.nets import AttentionUnet

# --- Configuration: Point to your new input directory ---
# This assumes your training output was saved and is now an input source.
INPUT_DIR = '/kaggle/input/sb23-2/'
PROCESSED_DATA_DIR = os.path.join(INPUT_DIR, 'BRATS/processed')
MODEL_PATH = os.path.join(INPUT_DIR, 'best_model_fold_0.pth') # Assuming fold 0 model

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- 1. Define the Model Architecture (must be identical to training) ---
model = AttentionUnet(
    spatial_dims=3,
    in_channels=4,
    out_channels=3,
    channels=(16, 32, 64, 128, 256), # Use the same channels as during training
    strides=(2, 2, 2, 2),
).to(DEVICE)

# --- 2. Load the Trained Model Weights ---
if os.path.exists(MODEL_PATH):
    print(f"Loading model weights from: {MODEL_PATH}")
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval() # CRITICAL: Set model to evaluation mode
else:
    raise FileNotFoundError(f"Model file not found at {MODEL_PATH}. Please check the path.")

# --- 3. Define Helper Function for Post-Processing ---
def remove_small_lesions(pred_mask_np, min_size_map):
    processed_mask = np.zeros_like(pred_mask_np)
    for c in range(pred_mask_np.shape[0]):
        channel_mask = pred_mask_np[c]
        if np.sum(channel_mask) > 0:
            labeled_mask = measure.label(channel_mask)
            min_size = min_size_map.get(c, 50)
            for region_label in range(1, np.max(labeled_mask) + 1):
                if np.sum(labeled_mask == region_label) > min_size:
                    processed_mask[c][labeled_mask == region_label] = 1
    return processed_mask


# ==============================================================================
# Part 2: Main Prediction and Visualization Function
# ==============================================================================
# This function encapsulates the entire process for a single patient ID.
# ------------------------------------------------------------------------------

def predict_and_visualize_3d(patient_id, model, device, data_dir):
    """
    Loads data for a patient, runs inference, and creates a side-by-side
    3D visualization of the ground truth and the prediction.
    """
    print(f"\n--- Processing Patient ID: {patient_id} ---")
    filepath = os.path.join(data_dir, f"{patient_id}.npz")

    if not os.path.exists(filepath):
        print(f"Error: Data for patient '{patient_id}' not found at '{filepath}'")
        return

    # --- 1. Load Data ---
    with np.load(filepath) as data:
        image = torch.from_numpy(data['image'].astype(np.float32))
        true_mask_np = data['mask']

    # --- 2. Run Inference ---
    input_tensor = image.unsqueeze(0).to(device)
    with torch.no_grad(), torch.cuda.amp.autocast():
        prediction_logits = model(input_tensor)

    # --- 3. Post-Process Prediction ---
    pred_mask_raw = (torch.sigmoid(prediction_logits).cpu().squeeze(0) > 0.5).numpy()
    min_lesion_sizes = {0: 100, 1: 75, 2: 50}  # WT, TC, ET (use same as validation)
    pred_mask_postprocessed = remove_small_lesions(pred_mask_raw, min_lesion_sizes)
    
    # --- 4. Create Side-by-Side 3D Plot ---
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'surface'}, {'type': 'surface'}]],
        subplot_titles=('Ground Truth', 'Model Prediction')
    )

    colors = ['blue', 'yellow', 'red']
    names = ["Whole Tumor (WT)", "Tumor Core (TC)", "Enhancing Tumor (ET)"]
    opacities = [0.2, 0.4, 0.7]

    def create_mesh_traces(mask_3d, names, colors, opacities):
        traces = []
        for i, (name, color, opacity) in enumerate(zip(names, colors, opacities)):
            if np.sum(mask_3d[i]) > 0:
                verts, faces, _, _ = measure.marching_cubes(mask_3d[i], level=0.5)
                x, y, z = verts.T
                traces.append(go.Mesh3d(
                    x=x, y=y, z=z,
                    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                    color=color, opacity=opacity, name=name, showlegend=True
                ))
        return traces

    gt_traces = create_mesh_traces(true_mask_np.astype(np.uint8), names, colors, opacities)
    if gt_traces:
        for trace in gt_traces:
            fig.add_trace(trace, row=1, col=1)

    pred_traces = create_mesh_traces(pred_mask_postprocessed, names, colors, opacities)
    if pred_traces:
        for trace in pred_traces:
            fig.add_trace(trace, row=1, col=2)

    # --- 5. CORRECTED: Configure Layout and Camera ---
    # Define a single camera view dictionary
    camera = dict(eye=dict(x=1.5, y=1.5, z=1.5))
    
    fig.update_layout(
        title_text=f'3D Segmentation Comparison for Patient: {patient_id}',
        # Apply camera settings to each scene individually
        scene1=dict(
            xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
            aspectratio=dict(x=1, y=1, z=1),
            camera=camera
        ),
        scene2=dict(
            xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
            aspectratio=dict(x=1, y=1, z=1),
            camera=camera # Use the same camera dict to link views
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(x=1.1)
    )

    fig.show()


# ==============================================================================
# Part 3: Run on a Sample Patient
# ==============================================================================
# Let's pick a patient from your processed data directory and visualize it.
# ------------------------------------------------------------------------------
try:
    # Get a list of available patient IDs from the input directory
    available_patients = [f.replace('.npz', '') for f in os.listdir(PROCESSED_DATA_DIR)]
    
    if available_patients:
        # You can replace this with any ID from your dataset
        sample_patient_id = available_patients[25] # Using the 26th patient for variety
        
        predict_and_visualize_3d(
            patient_id=sample_patient_id,
            model=model,
            device=DEVICE,
            data_dir=PROCESSED_DATA_DIR
        )
    else:
        print(f"No processed files found in {PROCESSED_DATA_DIR}. Cannot run visualization.")

except Exception as e:
    print(f"An error occurred: {e}")